In [1]:
from __future__ import annotations

import uuid
from enum import Enum
import errno
import sys
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

from debugpy.common.log import LEVELS

In [2]:
from identifier import Id, CompositeId
from meta import Run, Dir, File
from db import Db

In [3]:
db_name = 'test-db-for-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

the_db = Db.open(path, readonly=readonly, create=create)
env = the_db.env

In [4]:
import identifier
reload(identifier)
from identifier import Id, CompositeId

In [5]:
import db
reload(db)
from db import Db

In [ ]:
# show lmdb env info

env.flags(), env.path(), env.info()

In [43]:
env.close()

In [16]:
# list all key-value pairs in the db, using lmdb's env

with env.begin(
    write=False,
    # buffers= ?
) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            # if key.startswith(b'r:'):
            #     print(key, value)
            print(f'key={key}', f'value={value}')

key=b'd:\x00*\x00\x01' value=b'\x08\x01\x12\x04\x00*\x00\x01"\x01.-f\xe6@F2\x101111222233334444:\x105555666677778888'
key=b'd:\x00*\x00\x02' value=b'\x08\x01\x10*\x18\x02*\x01.5f\xe6@F'
key=b'dhd:' value=b'\x00*\x00\x02'
key=b'dhd:UUffww\x88\x88\x11\x11""33DD' value=b'\x00*\x00\x01'
key=b'dhf:' value=b'\x00*\x00\x02'
key=b'dhf:\x11\x11""33DDUUffww\x88\x88' value=b'\x00*\x00\x01'
key=b'f:\x00*\x00\x01\x00\x01' value=b'\x08\x01\x10*\x18\x01(\x018\xb2\xf2\x19'
key=b'f:\x00*\x00\x01\x00\x02' value=b'\x08\x01\x12\x06\x00*\x00\x01\x00\x02\x1a\tsome_file%\xcd\xc0\xa9E(\xb2\xf2\x192 abcd1234abcd1234abcd1234abcd1234'
key=b'fh:fedcba9876543210' value=b'\x00*\x00\x01\x00\x01'
key=b'fh:\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124' value=b'\x00*\x00\x01\x00\x01'
key=b'r:\x00*' value=b'\x08\x01\x10*\x1a\x11C:\\some\\start\\dir" run from jupyter for development*\x05koppa5\x9a\x99\x99?=\x9a\x99\x99?B\x0binitializedP\x08X\x18b\x0b\n\x04fld1\x12\x03blab\x16\n\x04fld2\x12\x0esan 64\r\nder 19'


In [9]:
CompositeId.bytes_to_bytes(b'\x00*\x01\x01\x00\x01')

(b'\x00*', b'\x01\x01', b'\x00\x01')

In [6]:
# create a Run obj

run42 = Run(
    id=Id(42),
    uuid=uuid.uuid4(),
    path=Path('C:\\some\\start\\dir'),
    description='run from jupyter for development',
    platform='lekker plat',
    start_time=1.2,
    end_time=2.4,
    duration=1.2,
    status='initialized',
    root_dir=None,
    num_dirs=8,
    num_files=24,
    extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
    tags=['san', 'der']
)

In [8]:
run42.id, run42.path, run42.root_dir, run42.platform

(Id(42), WindowsPath('C:/some/start/dir'), None, 'lekker plat')

In [9]:
run42.platform = "hoppa"

In [7]:
# create a Dir obj

dir1 = Dir(
    run = run42,
    id = CompositeId(run42.id),
    path=Path('thisisadirname'),
    path_repr='reppur',
    timestamp=12345.6,
    files_hash='1111222233334444',
    dirs_hash='5555666677778888',
    all_hash='9999aaaabbbbcccc',
)

In [8]:
dir1.id, dir1.id.to_hex()

(CompositeId(42, 1), '002a0001')

In [9]:
run42.root_dir = dir1

In [14]:
# create a File obj

file1 = File(
    run = run42,
    id = CompositeId(dir1.id),
    name = 'some_file',
    dir=dir1,
    creation_time=5432.1,
    length=424242,
    hash='abcd1234' * 4,
)

In [14]:
file1.id, file1.id.to_hex(), file1.dir.id

(CompositeId(42, 1, 1), '002a00010001', CompositeId(42, 1))

In [ ]:
# Store Run obj
roe = run42.root_dir
the_db.put_run(run42)

In [20]:
# Restore Run obj from db

run_id = Id(42)
rrun42 = the_db.get_run(run_id)
rrun42.id


Id(42)

In [22]:
rrun42.id.to_hex() == run42.id.to_hex()

True

In [23]:
run42.root_dir, rrun42.root_dir


(Dir(run=Run(id=Id(42), path=WindowsPath('C:/some/start/dir'), description='run from jupyter for development', platform='hoppa', start_time=1.2, end_time=2.4, duration=1.2, root_dir=..., extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'}, status='initialized', num_dirs=8, num_files=24, error=None), id=CompositeId(42, 1), name='dir', path=WindowsPath('.'), path_repr='dir', timestamp=12345.6, parent=None, num_files=-1, num_dirs=-1, file_ids=[], dir_ids=[], file_hashes=[], dir_hashes=[], files_hash='', dirs_hash='', all_hash=''),
 None)

In [11]:
# store Dir obj

the_db.put_dir(dir1)

Stored dir (42, 1)


In [15]:
# store File obj

the_db.put_file(file1)
# len(file1.hash)

Stored file (42, 1, 2)


In [39]:
p = Path('..\\mydb\\data.mdb')
p, str(p), p.as_posix(), str(p.absolute()), p.resolve(strict=False)

(WindowsPath('../mydb/data.mdb'),
 '..\\mydb\\data.mdb',
 '../mydb/data.mdb',
 'C:\\Users\\skrus\\Dropbox\\devel\\find image duplicates\\imagedups\\jupyter\\..\\mydb\\data.mdb',
 WindowsPath('C:/Users/skrus/Dropbox/devel/find image duplicates/imagedups/mydb/data.mdb'))

In [19]:
b = b'prefix:abcd:'
b.split(b':')

[b'prefix', b'abcd', b'']

In [21]:
bytes.fromhex('')

b''

In [25]:
import uuid
u: uuid.UUID = uuid.uuid4()
u

UUID('9069ff3c-32ea-4a26-a96c-c490714bf3db')

In [30]:
len(u.bytes)

16

In [31]:
u2 = uuid.UUID(bytes=u.bytes)
u2

UUID('9069ff3c-32ea-4a26-a96c-c490714bf3db')

In [32]:
u2 == u

True

In [33]:
id(u2), id(u)

(2710863957520, 2710125530640)

In [10]:
the_db.max_run_id()


0